## Tutorial

# Calculate and Compare Canopy Water Content

## Second of two notebooks

### Authors: Hannah Rieder, Randi Neff, Bridget Hass

In this tutorial, we will learn how to evaluate forest health using a calculation of the Canopy Water Content (CWC) from individual tiles at the Soaproot Saddle (SOAP) field site in the Sierra National Forest in California. The hyperspectral data for the CWC calculation comes from the National Ecological Observatory Network's (NEON) Level 3 Spectrometer orthorectified surface directional reflectance - mosaic data product and the Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product.

## The objectives of this tutorial (divided between two notebooks) are to:

* Use co-located data from NEON and EMIT
* Calculate Canopy Water Content (CWC) from NEON and EMIT hyperspectral data
* Evaluate CWC data at different scales
* Compare between burned and unburned areas

DATA The data provided with this tutorial were derived from existing code at:

* NEON Spectrometer orthorectified surface bidirectional reflectance data.
* Shapefiles for Creek fire boundary and NEON burned and unburned tiles which are found in the DATA folder.
* EMIT L2A Estimated Surface Reflectance granule(s) that cover the NEON burned and unburned tiles.
* Land Processes Distributed Active Archive Center (LP DAAC).

Additional data will be downloaded programmatically within this tutorial.

## What we should have after completing notebook 1:

* two EMIT cropped datasets (one for the burned tile and one for the unburned tile) exported to netcdf files
* two NEON reflectance datasets (one for the burned tile and one for the unburned tile). These datasets will already have been converted from hdf5 format into xarray, have the scale factor applied, have bad bands set to NaN, have necessary data types turned from float64 to float32, and be exported to netcdf files
* **make sure we have all 4 scenarios**

## Tutorial Outline for Notebook 2 - Canopy Water Content Comparison

1. Open NEON and EMIT Reflectance Data
2. Calculate Canopy Water Content (CWC)
3. Compare CWC Datasets

## Reference/credit to:
the [3 Equivalent Water Thickness/Canopy Water Content from Imaging Spectroscopy Data](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html).

In [1]:
# Import Packages
import os, sys #python module to create and acces file paths
# Some cells may generate warnings that we can ignore.
# Comment below lines to see.
import warnings
warnings.filterwarnings('ignore')

import numpy as np #work with multi-dimensional arrays
import xarray as xr #work with labelled multi-dimenstional arrays
from osgeo import gdal #work with raster and vector geospatial data
import rasterio as rio #work with geospatial raster data
import rioxarray as rxr #work with raster arrays
from matplotlib import pyplot as plt #plotting data
import hvplot.xarray #plot multi-dimensional arrays
import hvplot.pandas #plot DataFrames/Series
import pandas as pd #work with DataFrames
import geopandas as gpd #work with geospatial shapefiles
import earthaccess #search for, download, & stream NASA earth data

from modules.emit_tools import emit_xarray #open EMIT datasets into xarray.Dataset
from modules.ewt_calc2 import calc_ewt, calc_ewt_neon #canopy water content fxn
from scipy.optimize import least_squares #nonlinear least-squares

from modules.test_functions import data_download_tracker, surfrfl_hvplot_image
import neonutilities as nu #work with NEON reflectance data
import h5py #work with NEON reflectance data

### 1. Open NEON and EMIT Reflectance Data

In [2]:
# Define filepaths to the cropped NEON reflectance NetCDF files
neon_burn_fp = ("../data/REFL/neon_burn_refl_float32.nc")

#neon_unburn_fp = ("../data/REFL/neon_unburn_refl.nc")

In [3]:
# Define filepaths to the cropped EMIT reflectance NetCDF files
emit_burn_fp = ("../data/REFL"
                "/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burn.nc")
# emit_unburn_fp = ("../data/REFL/emit_unburn_refl.nc")

In [4]:
# Open NetCDF NEON & EMIT burned and unburned datasets
neon_burn_ds = xr.open_dataset(neon_burn_fp, decode_coords="all")
#neon_unburn_ds = xr.open_dataset(neon_unburn_fp, decode_coords="all")

emit_burn_ds = xr.open_dataset(emit_burn_fp, decode_coords="all")
# emit_unburn_ds = xr.open_dataset(emit_unburn_fp, decode_coords="all")

Check datasets

why - what are we checking for??

acknowledge that this isn't reproducible but for this purpose we're just opening them to make sure they look right before we complete the CWC calculation.

In [5]:
# Check neon_burn_ds
neon_burn_ds

<xarray.Dataset> Size: 2GB
Dimensions:           (y: 1000, x: 1000, wavelengths: 426)
Coordinates:
  * x                 (x) float64 8kB 2.98e+05 2.98e+05 ... 2.99e+05 2.99e+05
  * y                 (y) float64 8kB 4.1e+06 4.1e+06 ... 4.101e+06 4.101e+06
    fwhm              (wavelengths) float32 2kB ...
    good_wavelengths  (wavelengths) float32 2kB ...
    spatial_ref       int32 4B ...
  * wavelengths       (wavelengths) float32 2kB 383.9 388.9 ... 2.512e+03
Data variables:
    reflectance       (y, x, wavelengths) float32 2GB ...
Attributes:
    no_data_value:     -9999.0
    scale_factor:      10000.0
    bad_band_window1:  [1340 1445]
    bad_band_window2:  [1790 1955]
    projection:        +proj=UTM +zone=11 +ellps=WGS84 +datum=WGS84 +units=m ...
    spatial_ref:       PROJCS["WGS_1984_UTM_Zone_11N",GEOGCS["GCS_WGS_1984",D...
    EPSG:              32611

In [6]:
# Check neon_unburn_ds
#neon_unburn_ds

In [7]:
# Check emit_burn_ds
emit_burn_ds

<xarray.Dataset> Size: 457kB
Dimensions:           (latitude: 18, longitude: 22, wavelengths: 285)
Coordinates:
  * wavelengths       (wavelengths) float32 1kB 381.0 388.4 ... 2.493e+03
    fwhm              (wavelengths) float32 1kB ...
    good_wavelengths  (wavelengths) float32 1kB ...
  * latitude          (latitude) float64 144B 37.03 37.03 37.03 ... 37.03 37.02
  * longitude         (longitude) float64 176B -119.3 -119.3 ... -119.3 -119.3
    elev              (latitude, longitude) float32 2kB ...
    spatial_ref       int32 4B ...
Data variables:
    reflectance       (latitude, longitude, wavelengths) float32 451kB ...
Attributes: (12/40)
    ncei_template_version:             NCEI_NetCDF_Swath_Template_v2.0
    summary:                           The Earth Surface Mineral Dust Source ...
    keywords:                          Imaging Spectroscopy, minerals, EMIT, ...
    Conventions:                       CF-1.63
    sensor:                            EMIT (Earth Surface Mineral Dust Sourc...
    instrument:                        EMIT
    ...                                ...
    spatial_ref:                       GEOGCS["WGS 84",DATUM["WGS_1984",SPHER...
    geotransform:                      [-1.19978439e+02  5.42232520e-04 -0.00...
    day_night_flag:                    Day
    title:                             EMIT L2A Estimated Surface Reflectance...
    granule_id:                        EMIT_L2A_RFL_001_20230731T205320_23212...
    Orthorectified:                    True

In [8]:
# Check emit_unburn_cs
#emit_unburn_ds

### 2. Calculate Canopy Water Content (CWC)

#### Define necessary files

ADD NOTES ABOUT WHAT THE K_LIQUID_WATER_ICE.CSV IS AND WHERE TO GET THE K_LIQUID... FILE.

In [9]:
#define file path to k_liquid_water_ice.csv file
#this .csv was originally here: C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\data\SOAP\EMIT
#moved it to this filepath: C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\notebooks\exploratory\data
#after I got an error from the calc_ewt fxn that it couldn't find the .csv file.

# explain why this file is needed and that it has to be in this location
wp_fp = ("../data/k_liquid_water_ice.csv")

# Read k_liquid_water_ice.csv file into a DataFrame
k_wi = pd.read_csv(wp_fp)

# Check k_wi DataFrame
k_wi.head()

,wvl_1,T = 22°C,wvl_2,T = -8°C,wvl_3,T = -25°C,wvl_4,T = -7°C,wvl_5,T = 25°C (H),wvl_6,T = 20°C,wvl_7,T = 25°C (S),Index
0,666.7,2.470000e-08,NaN,NaN,NaN,NaN,660.0,1.660000e-08,650.0,1.640000e-08,650.0,1.870000e-08,650.12971,1.674130e-08,0
1,667.6,2.480000e-08,NaN,NaN,NaN,NaN,670.0,1.890000e-08,675.0,2.230000e-08,651.0,1.890000e-08,654.63616,1.777420e-08,1
2,668.4,2.480000e-08,NaN,NaN,NaN,NaN,680.0,2.090000e-08,700.0,3.350000e-08,652.0,1.910000e-08,660.69347,1.939950e-08,2
3,669.3,2.520000e-08,NaN,NaN,NaN,NaN,690.0,2.400000e-08,725.0,9.150000e-08,653.0,1.940000e-08,665.27314,2.031380e-08,3
4,670.2,2.530000e-08,NaN,NaN,NaN,NaN,700.0,2.900000e-08,750.0,1.560000e-07,654.0,1.970000e-08,669.88461,2.097930e-08,4


The function below gets the desired data from the csv file

The fxn below "uses least squares optimization to minimize the residuals of our Beer-Lambert Model and find a likely path length of liquid water" (from existing NASA EMIT CWC notebook).

#### Calculate CWC using the calc_ewt function imported in the beginning

In [10]:
# Learn about calc_ewt function
help(calc_ewt)

Help on function calc_ewt in module modules.ewt_calc2:

calc_ewt(filepath: str, outdir: str, n_cpu: int = 7, ewt_detection_limit: float = 0.5, return_cwc: bool = False, is_emit: bool = True) -> None
    This function will calculate the equivalent water thickness (EWT) or canopy water content (CWC) from an EMIT .nc reflectance
    file using `ray` for parallelization, orthorectify if necessary, and write a cloud-optimized geotiff output.



In [11]:
# Learn about neon_calc_ewt function
help(calc_ewt_neon)

Help on function calc_ewt_neon in module modules.ewt_calc2:

calc_ewt_neon(filepath: str, outdir: str, n_cpu: int = 7, ewt_detection_limit: float = 0.5, return_cwc: bool = False) -> None
    This function will calculate the equivalent water thickness (EWT) or canopy water content (CWC) from a NEON .nc reflectance
    file using `ray` for parallelization and write a cloud-optimized geotiff output. The NEON data are already be orthorectified
    and need to have a 'spatial_ref' as part of the ds.variables.keys().



In [12]:
# Set output directory where results of CWC function will be stored
out_dir = "../data/CWC/"

#original out dir:
#emit_out_dir = "../../../data/SOAP/EMIT/CWC/"
#add neon output directory here too? maybe just go back to one output directory?
# make consistent w/ where the .nc files are coming from, also depends on whether file names for outputs are clear
# possible NEW out dir: emit_out_dir = "../data/output/EMIT/CWC/"

Below is non-conditional code to calculate CWC using the EMIT reflectance data cropped to the burned tile (emit_burn_fp) and unburned tile (emit_unburn_fp). If the %%time isn't the first thing in the cell, it doesn't work. Need to investigate the %%time code and see how necessary it is.

In [ ]:
%%time
emit_burn_cwc_ds = calc_ewt(
    # Burned EMIT dataset file path
    emit_burn_fp,
    out_dir,
    ewt_detection_limit=1.5,
    return_cwc=True
)

# View emit_burn_cwc_ds 
emit_burn_cwc_ds

2025-07-25 13:02:59,751	ERROR services.py:1355 -- Failed to start the dashboard , return code 1
2025-07-25 13:02:59,751	ERROR services.py:1380 -- Error should be written to 'dashboard.log' or 'dashboard.err'. We are printing the last 20 lines for you. See 'https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#logging-directory-structure' to find where the log file is.
2025-07-25 13:02:59,751	ERROR services.py:1390 -- Couldn't read dashboard.log file. Error: [Errno 2] No such file or directory: 'C:\\Users\\riede\\AppData\\Local\\Temp\\ray\\session_2025-07-25_13-02-57_300434_44264\\logs\\dashboard.log'. It means the dashboard is broken even before it initializes the logger (mostly dependency issues). Reading the dashboard.err file which contains stdout/stderr.
2025-07-25 13:02:59,766	ERROR services.py:1424 -- Failed to read dashboard.err file: [Errno 2] No such file or directory: 'C:\\Users\\riede\\AppData\\Local\\Temp\\ray\\session_2025-07-25_13-02-57_300434

In [ ]:
# %%time
# emit_unburn_cwc_ds = calc_ewt(
#     # Unburned EMIT dataset file path
#     emit_unburn_fp,
#     out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

# # View emit_unburn_cwc_ds 
# emit_unburn_cwc_ds

Below is non-conditional code to calculate CWC using the NEON reflectance data cropped to the burned tile (neon_burn_fp) and unburned tile (neon_unburn_fp). 

In [ ]:
%%time
neon_burn_cwc_ds = calc_ewt_neon(
    # Burned NEON dataset file path
    emit_burn_fp,
    out_dir,
    ewt_detection_limit=1.5,
    return_cwc=True
)

# View neon_burn_cwc_ds 
neon_burn_cwc_ds

In [ ]:
# %%time
# neon_unburn_cwc_ds = calc_ewt_neon(
#     # Unburned NEON dataset file path
#     neon_unburn_fp,
#     emit_out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

# # View neon_unburn_cwc_ds 
# neon_unburn_cwc_ds

In [ ]:
# Start to a possible conditional statement for CWC calculation
cwc_soap_burn_fp = r'C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\data\SOAP\EMIT\CWC\EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burned_CWC.nc'
#if cwc has already been calculated and saved to a netcdf file,
if os.path.exists(cwc_soap_burn_fp):
    print('path exists')
    #open the CWC netcdf file and
    cwc_ds = xr.open_dataset(cwc_soap_burn_fp, decode_coords="all")
    #cwc_ds = emit_xarray(cwc_filepath)
    #display the CWC dataset
    display(cwc_ds)
else:
    print('path does not exist, calculating CWC...')
    #commented the %%time out b/c it threw an error, CWC still calculated w/o it for the burned tile
    #%%time
    emit_burn_cwc_ds = calc_ewt(
        #burned emit dataset
        emit_burn_fp,
        out_dir,
        ewt_detection_limit=1.5,
        return_cwc=True
    )
    display(emit_burn_cwc_ds)

# The above code all works, it just needs to be changes so it is reproducible and not specific to my file paths!

In [ ]:
# %%time
# emit_burn_cwc_ds = calc_ewt(
#     #burned emit dataset
#     emit_burn_fp,
#     emit_out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

# # View emit_burn_cwc_ds 
# emit_burn_cwc_ds

### 2.1 Visualize CWC Datasets

In [ ]:
# Plot CWC of the SOAP burned tile using surfrfl_hvplot_image fxn
surfrfl_hvplot_image(
    emit_burn_cwc_ds,
    plottitle=f"SOAP Burned Tile {emit_burn_cwc_ds.cwc.long_name} ({emit_burn_cwc_ds.cwc.units}) July 31, 2023",
clabel="Canopy Water Content (g/cm^2)")

In [ ]:
#export CWC datasets to NetCDF files so we don't have to run the CWC calculations again
# emit_burn_cwc_ds.to_netcdf("../../../data"
#            "/SOAP"
#            "/EMIT"
#            "/CWC/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burned_CWC.nc")

### 3. Compare CWC Datasets

Start w/ histogram comparisons of values - see bridget's 07 notebook. Bridget also created som eKDE (Kernal density plots)

Then think about rescaling and then finding the difference

